# CSE2530 Computational Intelligence
## Assignment 3: Reinforcement Learning

<div>
    
|    Group   |           1          |
|------------|----------------------|
| Junhan Chong  |        6153283       |
| Jack Bergmann  |        6200850       |
| Madhav Tiwari  |        6141870       |
| Ben Chen  |        6543405       |

#### Imports

In [1]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import matplotlib.pyplot as plt
import numpy as np
import random
from typing import Dict, List
from tqdm import tqdm

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Agent import Agent
from Maze import Maze
from QTable import QTable
from State import State
from Action import Action

## 2. Navigating in the Supermarket
### 2.1 Development
#### Question 1

In [2]:
class ExplorationStrategy:
    def __init__(self, q_table: QTable):
        self.q_table = q_table

    def random(self, agent: Agent, maze: Maze):
        """
        The random exploration strategy selects a random action uniformly at random
        from the set of all valid actions.
        """
        # find all possible actions for the particular state that the agent is in
        valid_as = agent.get_valid_actions()
        # choose an action from the list of valid actions randomly, with uniform distribution
        chosen_action = np.random.choice(valid_as)
        # take the chosen action
        return chosen_action

        


    def e_greedy(self, agent: Agent, maze: Maze, eps: float):
        """
        The e-greedy exploration strategy selects a random action with probability eps,
        and the action with highest q-value with probability 1 - eps. A value of epsilon
        close to 0 favours exploitation, while a value close to 1 favours exploration.
        """
        # find all possible actions for the particular state that the agent is in
        valid_as = agent.get_valid_actions()
        s = agent.get_state()

        # find the corresponding q values for the possible actions
        q_vals = [self.q_table.get_q(s, a) for a in valid_as]

        # find the actions with largest q value(s)
        max_q = np.max(q_vals)
        max_q_actions = [a for a, q in zip(valid_as, q_vals) if q == max_q]

        chosen_action = None
        # with probability of 1 - eps, select randomly from actions with the largest q values
        if (np.random.uniform() > eps):
            chosen_action = np.random.choice(max_q_actions)
        # with probability of eps, select randomly from the list of possible actions
        else:
            not_max = [a for a in valid_as if a not in max_q_actions]
            if len(not_max) == 0:
                np.random.choice(valid_as)
            else:
                chosen_action = np.random.choice(not_max)
        # take the chosen action
        return chosen_action



    def boltzmann(self, agent: Agent, maze: Maze, temperature: float):
        """
        The Boltzmann exploration strategy assigns a probability to each action based on its estimated q-values.
        A large value of the temperature encourages exploration, and as the temperature declines over time,
        exploitation is favoured. 
        """
        # find all possible actions for the particular state that the agent is in
        valid_as = agent.get_valid_actions(maze)
        s = agent.get_state(maze)

        # find the corresponding q values for the possible actions
        q_vals = [self.q_table.get_q(s, a) for a in valid_as]
        # subtract max q val for numerical stability
        q_vals = np.array(q_vals)
        q_vals = q_vals - np.max(q_vals)

        # guard against zero-division
        if temperature <= 0:
            # apply greedy if temperature is zero
            max_q = np.max(q_vals)
            max_q_actions = [a for a, q in zip(valid_as, q_vals) if q == max_q]
            chosen_action = np.random.choice(max_q_actions)
            return chosen_action

        # calculate exp(Q(s,a) / T) for all possible actions
        boltzmann_probs = np.exp(q_vals / temperature)

        # find the sum and divide each term by the sum
        boltzmann_probs /= np.sum(boltzmann_probs)

        
        # choose an action with the boltzmann probabilities as the weights
        chosen_action = np.random.choice(valid_as, p=boltzmann_probs)

        # take the chosen action
        return chosen_action

<div>

**Explain what you did, including the initial selection of the hyper-parameter values. How do you deal with exploration/exploitation tradeoff?**

All three strategies start with finding the valid actions that the agent can take at its current state. This is because the probability of choosing an invalid action should be zero in all given strategies.

For random exploration, I chose an action randomly based on an uniform distribution between the available actions. There are no hyper-parameters for this strategy because all decisions are made with a random uniform probability between actions. Therefore, this exploration strategy would be used initially to maximise exploration while neglecting exploitation.

For $\epsilon$-greedy exploration, I found the corresponding Q values for the possible actions to use for the greedy property of this strategy. With the Q values, we can simply choose the action with the highest associated Q value by definition. However, this approach would only use exploitation without the use of exploration since at any given point, the agent deterministically chooses what action to take. Therefore, we introduce a hyperparameter: $\epsilon$, to introduce exploration. Increasing $\epsilon$ increases exploration by increasing the probability of a random action being chosen, naturally decreasing exploitation as well. Therefore, we must find a suitable value of $\epsilon$ to achieve a good balance of exploitation and exploration in order to avoid convergence at a local optima, while keeping the learning relatively fast and efficient.

For Boltzmann exploration, we assign weights to the different actions to be used as the probability distribution. After finding the corresponding Q values for all possible actions, we use the softmax formula to assign probabilities for the actions. However, just by using the softmax function to assign weights for probability means we cannot tune the strategy to encourage or discourage more exploitation. Therefore, we introduce a hyperparameter $\tau$ (true Boltzmann exploration) which influences the probability of all possible actions. As we increase $\tau$, the probabilities decrease exponentially to its magnitude (because of the exponentiation). Hence, decreasing $\tau$ leads to more exploitation (as the higher Q value actions would be assigned a larger the probability relative to the lower Q value actions) while increasing it will even out the probability distribution towards uniform distribution; As $\tau$ approaches the limit of 0, the approach becomes greedier and greedier, while $T$ approaching infinity would make the strategy choose actions from a uniform distribution.

In practice, we should choose hyperparameters that encourage exploration (large $\epsilon$, large $\tau$) initially, so the agent can learn about the environment and map out a sensible Q table. As the agent explores, we can decrease the hyperparameters to favour more and more exploitation.

Hence, we can use Boltzmann exploration with a large value of $\tau$ initially to encourage exploration, then gradually 'decay' the temperature over each episode until it reaches a threshold to increase exploitation gradually. Since we have four actions per state, with a reward of 10 at the terminal state (initially), our Q values will not be larger than 10. Therefore, we can choose a relatively small value for the temperature, such as $\tau = 1$. Then we can use a fixed percentage value such as 0.99 to slowly decay the temperature over episodes. These hyper-parameters can be optimised during training (with the validation set) to maximise efficiency while minimising the risk of early convergence at a local optima.

#### Question 2

In [7]:
# Create a Maze instance.
maze = Maze("./../data/easy_maze.txt")
maze.set_reward(x=9, y=9, reward=10)
maze.set_terminal(x=9, y=9)
# Create an Agent.
agent = Agent(start_x=0, start_y=0)
# Create a QTable.
states = maze.get_all_states()
actions = [Action(id) for id in ["up", "down", "left", "right"]]
q_table = QTable(states, actions)
# Create an ExplorationStrategy.
exploration_strategy = ExplorationStrategy(q_table)
# Create a learner.
params = {"lr": 0.7, "gamma": 0.9}
# learner = QLearning(q_table, params)

# Hyper-parameters.
n_episodes = 300
episode_lengths = []
episode_rewards = []

alpha = 0.1
discount = 0.9
decay = 0.99
temperature = 1
done = False

# repeat exploration #n_episodes amount of time
for episode in tqdm(range(n_episodes)):
    # put agent at the start
    agent.reset()

    #counter to check progress
    steps = 0
    total_r = 0

    # explore until terminal state
    while (agent.get_state(maze).done == False):
        # one exploration step
        current_state = agent.get_state(maze)
        chosen_action = exploration_strategy.boltzmann(agent, maze, temperature)
        next_state, r, done = agent.step(chosen_action, maze)

        # variables for Q table update calculations
        q_old = q_table.get_q(current_state, chosen_action)
        q_max_new = np.max([q_table.get_q(next_state, a) for a in agent.get_valid_actions(maze)])

        # update Q table
        q_table.set_q(current_state, chosen_action, 
                      q_old + alpha * (r + discount * q_max_new - q_old))
        
        # re-assign/increment control variables
        current_state = next_state
        total_r += r
        steps += 1
    
    # decay the temperature every episode
    temperature *= decay

    # append progress variables to progress tracker
    episode_rewards.append(total_r)
    episode_lengths.append(steps)



100%|██████████| 300/300 [00:03<00:00, 77.34it/s] 


<div>

**Explain the cycle in your report.**

The cycle consists of three steps until the agent reaches a terminal state:
1. **Take action step:** We decide on the action for the agent to take. We used the Boltzmann strategy, with an initial temperature of 1.0. We then take the chosen action, getting the new state and reward for the action (at the particular state) as a return value. <br>
*\<This step is necessary for the agent to explore the environment\>*
2. **Update Q table:** We take the Q values of the initial state and action that we've chosen, as well as the Q values for max state-action value for the next state. Using these, we calculate the new Q value for the initial state-action pair. <br>
*\<This step is necessary to learn the environment. Q table is what is used to exploit efficient paths\>*
3. **Update states and reward metrics:** We then update the current state variable to the new position of the agent, as well as incrementing the rewards and step count to use as a performance metric. <br>
*\<This step is necessary to evaluate progress and allow for a cycle\>*

This cycle can be repeated until the agent reaches the terminal state, at which a new episode will start. These three steps are sufficient for a complete trial/episode (agent exploring from the start to terminal state) as it can explore the environment and learn from it (Q table). Repeating this cycle over the episodes will lead to a convergence in the Q value according to the **Stochastic Approximation Theory**. Then we can improve the policy based on the values from the Q table.

#### Question 3

In [ ]:
# Create a Maze instance.
maze = Maze("./../data/easy_maze.txt")
maze.set_reward(x=9, y=9, reward=10)
maze.set_terminal(x=9, y=9)
# Create an Agent.
agent = Agent(start_x=0, start_y=0)
# Create a QTable.
states = maze.get_all_states()
actions = [Action(id) for id in ["up", "down", "left", "right"]]
q_table = QTable(states, actions)
# Create an ExplorationStrategy.
exploration_strategy = ExplorationStrategy(q_table)
# Create a learner.
params = {"lr": 0.7, "gamma": 0.9}
# learner = QLearning(q_table, params)

# Hyper-parameters.
n_episodes = 300
episode_lengths = []
episode_rewards = []

alpha = 0.1
discount = 0.9
decay = 0.99
temperature = 1
done = False

# global step tracker
total_steps = 0

# repeat exploration #n_episodes amount of time
for episode in tqdm(range(n_episodes)):
    # put agent at the start
    agent.reset()

    #counter to check progress
    steps = 0
    total_r = 0

    # explore until terminal state
    while (agent.get_state(maze).done == False):
        if (total_steps > 30000):
            print("30000 steps reached. Terminating current cycle")
            break
        # one exploration step
        current_state = agent.get_state(maze)
        chosen_action = exploration_strategy.boltzmann(agent, maze, temperature)
        next_state, r, done = agent.step(chosen_action, maze)

        # variables for Q table update calculations
        q_old = q_table.get_q(current_state, chosen_action)
        q_max_new = np.max([q_table.get_q(next_state, a) for a in agent.get_valid_actions(maze)])

        # update Q table
        q_table.set_q(current_state, chosen_action, 
                      q_old + alpha * (r + discount * q_max_new - q_old))
        
        # re-assign/increment control variables
        current_state = next_state
        total_r += r
        steps += 1
        total_steps += 1
    
    # decay the temperature every episode
    temperature *= decay

    # append progress variables to progress tracker
    episode_rewards.append(total_r)
    episode_lengths.append(steps)

    if (total_steps > 30000):
        print("30000 steps reached. Terminating current episode.")
        break



  4%|▍         | 13/300 [00:00<00:15, 18.43it/s]

30000 steps reached. Terminating current cycle
30000 steps reached. Terminating current episode.


<div>



#### Question 4

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 5

In [ ]:
class QLearning:

    def __init__(self, q_table: QTable, params: Dict[str, float]) -> None:
        self.q_table = q_table
        self.params = params

    def learn(self, possible_actions: List[Action], state: State, action: Action,
               next_state: State, reward: int, done: bool) -> None:
        pass

In [ ]:
class SARSA:

    def __init__(self, q_table: QTable, params: Dict[str, float]) -> None:
        self.q_table = q_table
        self.params = params
    
    def learn(self, state: State, action: Action, next_state: State, next_action: Action,
               reward: float, done: bool) -> None:
        pass

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

#### Question 6

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 7

<div style="background-color:#f1be3e">

_Write your answer here._

### 2.2 Optimization
#### Question 8

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 9

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 10

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 11

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 12

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.3 Introducing More Rewards
#### Question 13

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 14

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 15

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

## 3. Open Questions
### 3.1 Reflection
#### Question 17

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 18

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 19

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper
#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**